# 🤖 Meeting Intelligence POC
### BrightLane Retail – AI Automation Workflow

**Pipeline:**
1. Install dependencies
2. Set API key
3. Paste or load your transcript
4. Run AI summarisation (Claude)
5. Generate PowerPoint deck
6. Preview email draft

> ⚠️ **Human review is required before distributing any outputs.**

---
## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

packages = ["anthropic", "python-pptx"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ Dependencies ready.")

✅ Dependencies ready.


---
## Cell 2 — Imports & Colour Palette

In [3]:
import os
import json
import re
import datetime
from pathlib import Path
from IPython.display import display, HTML, FileLink

import anthropic
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN

# Colour palette
DARK_BLUE   = RGBColor(0x1A, 0x37, 0x5E)
ACCENT_BLUE = RGBColor(0x27, 0x6E, 0xB4)
LIGHT_GREY  = RGBColor(0xF5, 0xF5, 0xF5)
WHITE       = RGBColor(0xFF, 0xFF, 0xFF)
DARK_TEXT   = RGBColor(0x22, 0x22, 0x22)
MID_GREY    = RGBColor(0x88, 0x88, 0x88)

print("✅ Imports complete.")

✅ Imports complete.


---
## Cell 3 — Configuration

Set your API key and output folder here.

In [5]:
# ── EDIT THESE ────────────────────────────────────────────────────────────
ANTHROPIC_API_KEY = "************************************************"   # paste your key
OUTPUT_DIR        = "./AiAutomation"               # folder for deck + audit log
# ──────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not ANTHROPIC_API_KEY or ANTHROPIC_API_KEY.startswith("sk-ant-your"):
    print("⚠️  Please paste your real Anthropic API key above.")
else:
    print(f"✅ Config ready. Outputs will be saved to: {os.path.abspath(OUTPUT_DIR)}")

✅ Config ready. Outputs will be saved to: C:\Users\rober\Downloads\AiAutomation


---
## Cell 4 — Load Transcript

**Option A:** Paste transcript text directly into `TRANSCRIPT_TEXT`  
**Option B:** Point `TRANSCRIPT_FILE` to a `.txt` file — it will be loaded automatically.

In [7]:
# Option A: paste transcript directly (leave TRANSCRIPT_FILE as None)
TRANSCRIPT_TEXT = """
Synthetic Transcript – Retail AI Automation Discovery Call

Company: BrightLane Retail
Industry: Multi-location specialty retail / home goods / seasonal decor
Meeting Length: ~20 minutes
Participants:
- Sarah Kim (CEO, BrightLane Retail)
- Marcus Lee (Director of Operations, BrightLane Retail)
- Tina Patel (Ecommerce & Customer Experience Manager, BrightLane Retail)
- Alex Rivera (AI Automation Specialist / Consultant)

[00:00] Alex:
Thanks, everyone, for making time today. I know we’ve only got about 20 minutes, so maybe we can keep this pretty practical. My goal is really to understand where the biggest operational pain points are, what’s repetitive, what feels manual, and where an AI-enabled workflow might realistically help without creating a giant IT project. Does that sound right?

[00:18] Sarah:
Yeah, that’s exactly right. We are not looking for some huge enterprise transformation. We’re a growing retail company, but we’re still pretty lean. We have 14 stores, an ecommerce site, and a small corporate team, so we need things that are useful and not overly expensive.

[00:36] Alex:
Perfect. Why don’t we start with the business at a high level. Sarah, maybe just a quick overview of BrightLane and where you feel the friction most.

[00:45] Sarah:
Sure. BrightLane sells home goods, gifts, seasonal decor, and some private-label products. We’ve grown pretty fast over the last two years. That’s been great, but a lot of our processes still work like we’re a five-store business. So I’d say our biggest issues are internal reporting, inventory coordination, and customer service follow-up. We’ve layered tools on top of tools and now people are doing a lot of manual copying, checking, summarizing, emailing, that kind of thing.

[01:12] Alex:
Got it. And when you say internal reporting, what does that actually look like today?

[01:18] Marcus:
Honestly, kind of ugly. Every Monday I pull store performance data from our POS, then I pull ecommerce metrics from Shopify, and then I try to combine that with inventory notes from store managers. We also track labor in another system. None of it lives together. So I usually spend like three or four hours every Monday morning just assembling a “what happened last week” report for leadership.

[01:44] Alex:
And that report is going to who?

[01:46] Marcus:
Sarah, finance, district managers, and sometimes merchandising depending on what’s going on.

[01:53] Alex:
And is it mostly quantitative reporting or are you also adding written commentary?

[01:58] Marcus:
Both. That’s part of the problem. The numbers are one piece, but then store managers send emails or fill out forms with things like “we ran out of this item,” “customers kept asking for this,” “foot traffic was weird because of weather,” “we had a staffing issue on Saturday,” and I’m trying to summarize all that into something useful.

[02:18] Sarah:
And a lot of that context matters more than the raw sales number by itself.

[02:22] Alex:
Yeah, that makes sense. Tina, what about on the customer side?

[02:27] Tina:
Customer support and customer experience are messy in a different way. We get inquiries from email, website chat, and social DMs. A lot of them are pretty repetitive. Things like “where’s my order,” “when will this item be back in stock,” “can I return in-store if I bought online,” “do you have this in blue,” stuff like that. My team answers most of it manually. We do use templates, but not in a smart way. So we still have people reading, triaging, routing, and rewriting the same answers over and over.

[02:58] Alex:
Roughly how much volume are you talking?

[03:01] Tina:
Depends on seasonality. Normal month maybe 1,500 to 2,000 customer messages across channels. Holiday months a lot more. And not everything should be automated, obviously, but I think maybe half to two-thirds are repetitive enough that a better workflow could help.

[03:18] Alex:
Okay, that’s meaningful volume. What systems are currently in play across the business?

[03:24] Marcus:
Shopify for ecommerce. Our POS is Lightspeed in stores. We use Gorgias for customer support. Slack internally. Google Workspace for docs and sheets. Some managers still email instead of using the form we created. We’ve got a few reports in Looker Studio, but they’re not really the source of truth. And inventory is… semi-centralized. We have one inventory planning tool, but it’s not very well adopted.

[03:50] Sarah:
That’s a nice way of saying it.

[03:52] Marcus:
Yeah.

[03:54] Alex:
No worries, this is very normal. Are there any hard constraints I should know? Security, compliance, budget, team skill set, anything like that?

[04:04] Sarah:
Budget matters. I’m not opposed to paying for something useful, but I do not want a bloated system with five subscriptions and a consultant dependency forever. We’d prefer something our ops team can actually understand. We don’t have internal engineers. We have one technically savvy operations analyst, but that’s about it.

[04:23] Tina:
And brand voice matters a lot if anything is customer-facing. We don’t want robotic responses going out to customers without some control.

[04:31] Alex:
Makes sense. Human review for external responses is totally reasonable, especially in phase one. Marcus, if you could wave a magic wand and automate one thing first, what would it be?

[04:42] Marcus:
Weekly reporting, easily. If I could upload or connect the sales, support, and store notes data and have something generate a leadership summary with red flags, trends, and recommended follow-ups, that would save real time and probably improve quality.

[04:58] Alex:
How do store notes come in right now?

[05:00] Marcus:
Mostly a Google Form that goes to a Sheet, but some people still send emails or Slack messages. So I’m consolidating multiple formats.

[05:09] Alex:
Okay. That’s actually a good use case because even semi-structured notes can be normalized. Sarah, what would make that weekly report feel valuable instead of just “faster”?

[05:19] Sarah:
Good question. I’d want it to do more than summarize. I’d want it to flag what matters. Like, don’t just tell me Store 8 sales were down 7%. Tell me that Store 8 was down, they also had two staffing callouts, and three managers mentioned low stock in a high-margin item, so maybe that’s the issue. I want prioritization, not just compression.

[05:42] Alex:
That’s helpful. So not just summarization, but synthesis and maybe some light reasoning.

[05:47] Sarah:
Exactly. And ideally, not hallucinated nonsense.

[05:50] Alex:
Fair. Very fair.

[05:53] Tina:
And on the support side, my dream would be something that reads incoming messages, categorizes them, drafts a response using our approved policies and brand tone, and either auto-sends for very low-risk stuff or puts it in a queue for review.

[06:10] Alex:
Do you have documentation already for policies like returns, exchanges, shipping, damaged goods?

[06:15] Tina:
Yes, but it’s scattered. Some in help center articles, some in internal docs, some kind of tribal knowledge.

[06:23] Alex:
That’s common. The nice thing is that’s fixable. If we were designing this in phases, I’d probably separate internal operations reporting from customer-facing automation so we don’t mix risk levels too early.

[06:35] Sarah:
That sounds smart. I do not want the first project to accidentally email customers the wrong thing.

[06:40] Alex:
Totally. So let me pressure test value. Marcus, how many hours per month do you think are spent on weekly reporting and related follow-up?

[06:48] Marcus:
For me personally, maybe 15 to 20 hours a month. Then probably another 8 to 10 hours across district managers and merchandising because they’re clarifying things after the report goes out.

[07:00] Alex:
So maybe 25 to 30 hours of management time monthly on that process.

[07:05] Marcus:
Yeah, that feels right.

[07:07] Alex:
Tina, for support, how many hours would you estimate are spent on repetitive first-pass handling?

[07:13] Tina:
A lot. Probably 60 to 80 hours a month conservatively, maybe more during peak. Not all of that disappears with automation, but even a 25% reduction would matter.

[07:24] Alex:
Right. Especially if it improves consistency. Are there any upcoming business events or deadlines making this more urgent?

[07:31] Sarah:
We have holiday planning starting in about three months. I’d really like some kind of internal reporting automation in place before then, because Q4 is where all the cracks show up. If the customer service piece comes later, that’s fine, but the internal summary and action recommendations would be immediately useful.

[07:50] Alex:
That’s good prioritization. So if we were keeping phase one narrow, I’m hearing something like: ingest meeting recordings and weekly operational data, transcribe and summarize the meeting, extract key decisions, combine with uploaded reports or notes, then generate an executive summary, objectives, action items, and maybe a simple slide deck or email for leadership. Does that track?

[08:12] Marcus:
Yes, that’s very close to what I’d want.

[08:15] Sarah:
And rough cost visibility. Like if it notices something is broken, I want to know the likely impact or at least where we should investigate. It doesn’t need to be perfect, but some rough business framing would be helpful.

[08:27] Alex:
Interesting. So maybe not true financial modeling, but maybe rules-based estimates or impact bands.

[08:32] Sarah:
Exactly. “Potential revenue impact,” “potential staffing cost issue,” that sort of thing.

[08:37] Alex:
Got it. And on the meeting side specifically, are these mostly leadership calls, store ops calls, or vendor meetings?

[08:44] Marcus:
Usually weekly leadership calls and ops reviews. They’re on Google Meet most of the time. Sometimes Zoom. People leave with action items, but then there’s always disagreement afterward about what was actually decided.

[08:56] Tina:
Yes. We definitely have the classic “I thought Marcus was doing that” problem.

[09:01] Marcus:
That is true.

[09:03] Alex:
Okay, that’s actually a very good starting use case because meeting summarization plus action-item extraction is relatively contained. Are recordings already available automatically?

[09:12] Marcus:
Sometimes. Not always. We can turn it on more consistently if needed.

[09:16] Alex:
And would you be okay with a workflow where someone uploads the video or audio plus maybe a spreadsheet or notes file into a folder, then the system processes it and outputs a summary package?

[09:28] Sarah:
Yes. That’s fine for a first version. It does not need to be magical. It just needs to save time and produce something useful.

[09:35] Alex:
Love that. That’s the right attitude for phase one. In terms of output format, what would leadership actually consume? Email, slide deck, PDF, dashboard?

[09:44] Sarah:
Email first. Slides second. Dashboard maybe eventually, but I don’t think we need to start there.

[09:50] Marcus:
Agreed. If it could output a concise email and a 4- or 5-slide deck template, that would be great.

[09:56] Tina:
And if the deck could be editable, even better. Like Google Slides or PowerPoint, not something flattened.

[10:03] Alex:
Yep, that’s very doable.

[10:10] Marcus:
Messy. Different naming conventions, different managers, occasional typos, inconsistent structure in comments.

[10:17] Alex:
Okay. That suggests we’d want a workflow with explicit preprocessing and maybe some validation before sending everything into a language model. Are there any metrics you always care about in the weekly view?

[10:29] Sarah:
Sales by store, comp trends if possible, top out-of-stock items, returns spikes, staffing issues, and major customer complaints or recurring themes.

[10:41] Tina:
On the customer side, I’d add response time, refund requests, damaged shipment mentions, and product confusion. We often learn that product pages are unclear because support gets flooded with the same question.

[10:52] Alex:
That’s a great feedback loop opportunity actually. Support as a source of merchandising and ecommerce insight.

[10:58] Sarah:
Exactly. Right now those insights are trapped in inboxes.

[11:03] Alex:
Okay. And who would own the workflow internally once it exists?

[11:07] Marcus:
Probably operations initially. With Tina contributing on the support taxonomy if we expand into that area.

[11:14] Alex:
And do you prefer low-code tools where possible, or are you open to a lightweight custom application if it reduces recurring costs?

[11:21] Sarah:
I’d say practical over ideological. Low-code is fine if it’s stable and not ridiculously expensive. Lightweight custom is fine if it’s maintainable by someone who didn’t build it.

[11:32] Alex:
Good answer. So maybe a hybrid. Something like cloud storage, transcription API, LLM summarization, and a thin orchestration layer. Possibly no-code or low-code for notifications and routing, depending on the stack.

[11:45] Marcus:
That sounds reasonable. I don’t need fancy. I need dependable.

[11:49] Alex:
That should be on a T-shirt.

[11:52] Sarah:
Yes.

[11:54] Alex:
Let’s talk budget range just loosely, because that tends to narrow architecture choices. Are you imagining a few hundred a month, low thousands, or more?

[12:03] Sarah:
For an initial workflow, I’d like software costs to stay under maybe $1,500 per month if possible, excluding one-time setup. If it proves real ROI, then obviously we can expand.

[12:15] Alex:
That’s helpful. That budget is workable for a narrowly scoped phase one. Especially if volume is around, say, 20 meetings a month plus some reporting documents.

[12:24] Marcus:
Yeah, that’s about right. Maybe 20 to 25 meetings that matter enough to process.

[12:29] Alex:
And average meeting length?

[12:31] Marcus:
Usually 30 to 60 minutes. The weekly leadership one is around 45.

[12:36] Alex:
Great. That’s manageable. One thing I’d likely recommend is keeping the workflow very auditable. Meaning transcript stored, extracted action items stored, source links attached, and maybe a confidence note where the model is less certain. Would that be useful?

[12:51] Sarah:
Yes, a lot. I don’t want black-box outputs. I want people to be able to say, “why is this recommendation here?” and trace it back.

[12:59] Tina:
Especially if the tool starts suggesting actions based on support tickets or meeting discussion. It needs to show its work, basically.

[13:07] Alex:
Exactly. And for the slide deck, do you want it visually polished or just clear and editable?

[13:12] Sarah:
Clear and editable. Nobody needs AI-generated art here.

[13:16] Alex:
Music to my ears. Okay, so if I were mapping a first version, I’d likely say: user uploads recording and optional supporting documents into a folder, automation triggers transcription, transcript and documents are cleaned and chunked, model generates summary, objectives, action items, and estimated business impact, then outputs an email draft and a short slide deck. A human reviews, edits if needed, and sends. Does that feel aligned?

[13:42] Marcus:
Yes.

[13:43] Tina:
Yep.

[13:44] Sarah:
Yes, as long as the recommendations are grounded enough to be useful.

[13:49] Alex:
Understood. I’d probably handle that by combining prompt instructions with simple business rules. So for example, if out-of-stock mentions exceed a threshold and sales are down in the same category, the workflow can flag that as a likely merchandising issue rather than pretending the model has magical certainty.

[14:07] Sarah:
That’s good. I like the combination of AI plus rules. I don’t want “AI intuition” without evidence.

[14:13] Alex:
Very fair. What would success look like 60 days after launch?

[14:18] Marcus:
For me, cutting my Monday reporting time in half, at least. And having leadership trust the output enough that they rely on it.

[14:26] Tina:
For me, success would be getting cleaner visibility into recurring support themes, even before we automate responses. I think that alone would help us fix problems upstream.

[14:36] Sarah:
For me it’s two things: time savings and better decisions. If this just makes busywork faster, that’s nice, but I’d really like it to improve prioritization.

[14:47] Alex:
That’s a strong framing. Are there any reasons this would fail internally? Adoption, messy process ownership, lack of discipline around uploads?

[14:56] Marcus:
Yes to all of those, honestly. The biggest issue is consistency. If managers don’t use the agreed format or don’t upload materials on time, the workflow won’t have good inputs.

[15:08] Sarah:
But that’s also why I think a meeting-based workflow is a good starting point. Fewer people involved, more controlled inputs.

[15:14] Alex:
I agree. Start where the signal is strongest. Then expand outward.

[15:18] Tina:
And maybe use that first project to build trust before we touch anything customer-facing.

[15:23] Alex:
Exactly. One more question on infrastructure preferences. Do you care whether this lives in Google, Microsoft, AWS, something else?

[15:31] Marcus:
No strong preference from me.

[15:34] Sarah:
We’re already deep in Google Workspace operationally, so if outputs can land there easily, that helps. But I care more about reliability than brand.

[15:43] Alex:
That’s fair. So Google Drive or a shared folder for intake and outputs may be the simplest UX, even if the processing happens elsewhere.

[15:50] Sarah:
Right.

[15:51] Alex:
Would you want the final output also posted into Slack?

[15:54] Marcus:
Yes, probably a notification to a leadership channel saying the summary package is ready.

[16:00] Alex:
Easy enough. Are there any approval requirements? Like should the email or deck always require Marcus or Sarah approval before distribution?

[16:08] Sarah:
Yes. At least initially. I would say human approval is mandatory before anything goes out broadly.

[16:15] Alex:
Perfect. That lowers risk. Let me summarize back what I’m hearing and you can tell me if I missed anything. BrightLane wants a practical AI automation workflow, likely starting with recorded leadership or operations meetings. The system should transcribe the meeting, generate an executive summary, identify top objectives, extract action items and next steps, and ideally incorporate supporting notes or metrics where available. It should then produce an email-friendly output and a short editable slide deck. The company values cost control, explainability, human review, and maintainability. Longer term, there is a second opportunity around customer support triage and drafted responses, but phase one should focus on internal operations reporting and meeting intelligence. Did I get that right?

[16:59] Sarah:
Yes, that’s very accurate.

[17:01] Marcus:
Yep.

[17:02] Tina:
That captures it.

[17:04] Alex:
Great. In terms of next steps, I’d probably propose a small proof of concept around one recurring meeting type. Maybe take three historical meeting recordings, plus any notes or related weekly reports, and test how well the workflow can produce consistent summaries and action-item extraction. Then we’d evaluate quality, cost, and where humans need to stay in the loop.

[17:25] Sarah:
That sounds right. I’d also want some very basic cost estimate before we commit, even if it’s directional.

[17:31] Alex:
Absolutely. I can put together something with a lightweight architecture option and maybe one slightly more scalable option so you can compare.

[17:39] Marcus:
And if possible, call out likely failure points. Like poor transcript quality, inconsistent naming, duplicated action items, that kind of thing.

[17:47] Alex:
Definitely. Those are exactly the things to surface early. Any must-have deliverables from your perspective besides the summary and deck?

[17:55] Sarah:
I’d like the three recommended actions to be really business-oriented, not generic. So not “improve communication.” More like “investigate Store 8 stockout pattern in bedding category,” that kind of thing.

[18:06] Alex:
Specific, grounded, and operational. Got it.

[18:10] Tina:
And for the future support use case, if you could maybe note what would be needed to expand into that later, that would help me make the internal case.

[18:18] Alex:
Sure. I can include that as a phase-two note. Anything else before we wrap?

[18:24] Marcus:
I think that covers it.

[18:26] Sarah:
No, this was good. I appreciate that you’re thinking about something practical versus a science project.

[18:32] Alex:
That is always the goal. I’ll follow up with a simple architecture, rough monthly costs, a proof-of-concept recommendation, and some notes on risks and assumptions. Thanks, everyone.

[18:42] Tina:
Thanks.

[18:43] Marcus:
Thanks, Alex.

[18:44] Sarah:
Thanks....
"""

# Option B: load from file (set path, leave TRANSCRIPT_TEXT as empty string)
TRANSCRIPT_FILE = None   # e.g. "transcript.txt"

# ── Auto-select source ────────────────────────────────────────────────────
if TRANSCRIPT_FILE and Path(TRANSCRIPT_FILE).exists():
    transcript = Path(TRANSCRIPT_FILE).read_text(encoding="utf-8")
    print(f"[ingestion] Loaded from file: {TRANSCRIPT_FILE}")
elif TRANSCRIPT_TEXT.strip() and TRANSCRIPT_TEXT.strip() != "Paste your meeting transcript here...":
    transcript = TRANSCRIPT_TEXT.strip()
    print(f"[ingestion] Using pasted transcript ({len(transcript):,} characters).")
else:
    raise ValueError("No transcript found. Paste text into TRANSCRIPT_TEXT or set TRANSCRIPT_FILE path.")

print(f"   Word count (approx): {len(transcript.split()):,}")
print(f"   Preview: {transcript[:200]}...")

[ingestion] Using pasted transcript (20,799 characters).
   Word count (approx): 3,265
   Preview: Synthetic Transcript – Retail AI Automation Discovery Call

Company: BrightLane Retail
Industry: Multi-location specialty retail / home goods / seasonal decor
Meeting Length: ~20 minutes
Participants:...


---
## Cell 5 — AI Summarisation (Claude API)

Sends the transcript to Claude and returns structured JSON.

In [9]:
SYSTEM_PROMPT = """
You are an expert business analyst. You will receive a raw meeting transcript.
Your job is to extract structured information and return ONLY valid JSON —
no markdown fences, no preamble, no commentary.

Return exactly this JSON schema:
{
  "meeting_title": "string",
  "meeting_date": "string (ISO date or best guess)",
  "participants": ["list of names"],
  "executive_summary": "2-3 sentence paragraph summarising the meeting purpose, context, and outcome",
  "objectives": [
    {"title": "string (8 words max)", "detail": "string (1-2 sentences, specific and business-grounded)"},
    {"title": "string", "detail": "string"},
    {"title": "string", "detail": "string"}
  ],
  "action_items": [
    {"owner": "Name or Role", "action": "string (specific, operational)", "due": "string or TBD"},
    {"owner": "string", "action": "string", "due": "string"},
    {"owner": "string", "action": "string", "due": "string"}
  ],
  "next_steps": [
    "string (concrete next step 1)",
    "string (concrete next step 2)",
    "string (concrete next step 3)"
  ],
  "risks_and_flags": [
    "string (any risk, blocker, or concern raised)"
  ],
  "confidence_note": "string - note any areas where the transcript was ambiguous"
}

Rules:
- Objectives must be HIGH-LEVEL business goals (not tasks).
- Action items must be SPECIFIC and OPERATIONAL (not generic like improve communication).
- Do NOT hallucinate facts not present in the transcript.
- If information is missing, use null for that field.
""".strip()


def summarise_with_claude(transcript: str, api_key: str) -> dict:
    print("[ai] Calling Claude API...")
    client = anthropic.Anthropic(api_key=api_key)

    message = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2000,
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": f"Please analyse this meeting transcript:\n\n{transcript}"}
        ]
    )

    raw = message.content[0].text.strip()

    # Strip accidental markdown fences
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    try:
        data = json.loads(raw)
        print("✅ Structured data extracted successfully.")
        return data
    except json.JSONDecodeError as e:
        print(f"⚠️  JSON parse failed: {e}")
        print(f"Raw response snippet:\n{raw[:400]}")
        return {
            "meeting_title": "Meeting Summary",
            "meeting_date": str(datetime.date.today()),
            "participants": [],
            "executive_summary": raw[:500],
            "objectives": [], "action_items": [],
            "next_steps": [], "risks_and_flags": [],
            "confidence_note": "JSON parse failed — raw text used as fallback."
        }


# Run the API call
meeting_data = summarise_with_claude(transcript, ANTHROPIC_API_KEY)

# Pretty-print the extracted JSON
print("\n--- Extracted Data ---")
print(json.dumps(meeting_data, indent=2))

[ai] Calling Claude API...
✅ Structured data extracted successfully.

--- Extracted Data ---
{
  "meeting_title": "BrightLane Retail AI Automation Discovery Call",
  "meeting_date": "2024-01-15",
  "participants": [
    "Sarah Kim",
    "Marcus Lee",
    "Tina Patel",
    "Alex Rivera"
  ],
  "executive_summary": "BrightLane Retail, a 14-store specialty retail company, explored AI automation opportunities to address operational inefficiencies in reporting and customer service. The primary focus was weekly operations reporting, where Marcus currently spends 3-4 hours manually consolidating data from multiple systems. The team agreed to pursue a phased approach starting with meeting transcription and automated executive summaries, with customer support automation deferred to phase two.",
  "objectives": [
    {
      "title": "Automate Weekly Operations Reporting Workflow",
      "detail": "Reduce the 25-30 monthly hours spent consolidating POS data, ecommerce metrics, inventory notes, a

---
## Cell 6 — Preview Extracted Data

Renders the AI output as a readable HTML card inside the notebook.

In [11]:
def preview_data(d: dict):
    objectives_html = "".join(
        f"""<div style='background:#f0f4ff;border-left:4px solid #276eb4;
                        padding:10px 14px;margin:6px 0;border-radius:4px'>
               <strong style='color:#1a375e'>0{i+1}  {obj['title']}</strong><br>
               <span style='color:#444;font-size:13px'>{obj['detail']}</span>
             </div>"""
        for i, obj in enumerate(d.get("objectives", []))
    )

    actions_html = "".join(
        f"""<tr style='background:{'#f9f9f9' if i%2==0 else '#fff'}'>
               <td style='padding:8px 12px;border:1px solid #ddd;color:#276eb4;
                          font-weight:600'>{a.get('owner','—')}</td>
               <td style='padding:8px 12px;border:1px solid #ddd'>{a.get('action','—')}</td>
               <td style='padding:8px 12px;border:1px solid #ddd;white-space:nowrap'>{a.get('due','TBD')}</td>
             </tr>"""
        for i, a in enumerate(d.get("action_items", []))
    )

    steps_html = "".join(
        f"<li style='margin:4px 0;color:#333'>{s}</li>"
        for s in d.get("next_steps", [])
    )

    risks_html = "".join(
        f"<li style='margin:4px 0;color:#c0392b'>{r}</li>"
        for r in d.get("risks_and_flags", [])
    )

    html = f"""
    <div style='font-family:Arial,sans-serif;max-width:860px;border:1px solid #ddd;
                border-radius:8px;overflow:hidden'>

      <!-- Header -->
      <div style='background:#1a375e;padding:20px 24px'>
        <div style='color:#aac4e0;font-size:12px;margin-bottom:4px'>AUTO-GENERATED · REQUIRES HUMAN REVIEW</div>
        <div style='color:#fff;font-size:22px;font-weight:bold'>{d.get('meeting_title','Meeting Summary')}</div>
        <div style='color:#aac4e0;font-size:13px;margin-top:6px'>
          {d.get('meeting_date','')}  ·  {', '.join(d.get('participants',[]))}
        </div>
      </div>

      <div style='padding:20px 24px'>

        <!-- Executive Summary -->
        <h3 style='color:#1a375e;margin:0 0 8px'>Executive Summary</h3>
        <p style='color:#333;line-height:1.6;margin:0 0 20px'>{d.get('executive_summary','')}</p>

        <!-- Objectives -->
        <h3 style='color:#1a375e;margin:0 0 8px'>High-Level Objectives</h3>
        {objectives_html}

        <!-- Action Items -->
        <h3 style='color:#1a375e;margin:20px 0 8px'>Action Items</h3>
        <table style='width:100%;border-collapse:collapse;font-size:13px'>
          <tr style='background:#276eb4;color:#fff'>
            <th style='padding:8px 12px;text-align:left'>Owner</th>
            <th style='padding:8px 12px;text-align:left'>Action</th>
            <th style='padding:8px 12px;text-align:left'>Due</th>
          </tr>
          {actions_html}
        </table>

        <!-- Next Steps -->
        <h3 style='color:#1a375e;margin:20px 0 8px'>Next Steps</h3>
        <ol style='margin:0;padding-left:20px'>{steps_html}</ol>

        <!-- Risks -->
        {'<h3 style="color:#c0392b;margin:20px 0 8px">Risks &amp; Flags</h3><ul style="margin:0;padding-left:20px">' + risks_html + '</ul>' if risks_html else ''}

        <!-- Confidence note -->
        <div style='margin-top:20px;padding:10px 14px;background:#fffbe6;
                    border-left:4px solid #f0ad00;border-radius:4px;
                    font-size:12px;color:#555'>
          <strong>AI confidence note:</strong> {d.get('confidence_note','')}
        </div>

      </div>
    </div>
    """
    display(HTML(html))


preview_data(meeting_data)

Owner,Action,Due
Alex Rivera,"Deliver architecture proposal with two options (lightweight and scalable), rough monthly cost estimates under $1,500, proof-of-concept recommendation using three historical meetings, and documented risks/failure points",TBD
Alex Rivera,Include phase-two expansion notes for customer support automation use case to support internal business case development,TBD
Marcus Lee,Ensure consistent recording of leadership and operations meetings (Google Meet/Zoom) and establish discipline around timely upload of supporting materials,Before POC begins
Tina Patel,"Consolidate scattered policy documentation (returns, exchanges, shipping, damaged goods) from help center articles and internal docs into centralized reference for future automation",TBD


---
## Cell 7 — Generate PowerPoint Deck

In [13]:
def slide_header(slide, prs, title_text: str):
    hdr = slide.shapes.add_shape(1, 0, 0, prs.slide_width, Inches(1.1))
    hdr.fill.solid()
    hdr.fill.fore_color.rgb = DARK_BLUE
    hdr.line.fill.background()
    tb = slide.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(9), Inches(0.7))
    p = tb.text_frame.paragraphs[0]
    run = p.add_run()
    run.text = title_text
    run.font.size = Pt(22)
    run.font.bold = True
    run.font.color.rgb = WHITE


def add_title_slide(prs, d):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    bg = slide.shapes.add_shape(1, 0, 0, prs.slide_width, prs.slide_height)
    bg.fill.solid()
    bg.fill.fore_color.rgb = DARK_BLUE
    bg.line.fill.background()

    tb = slide.shapes.add_textbox(Inches(0.8), Inches(2.0), Inches(8.4), Inches(1.6))
    tf = tb.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    p.alignment = PP_ALIGN.LEFT
    run = p.add_run()
    run.text = d.get("meeting_title", "Meeting Summary")
    run.font.size = Pt(34)
    run.font.bold = True
    run.font.color.rgb = WHITE

    tb2 = slide.shapes.add_textbox(Inches(0.8), Inches(3.8), Inches(8.4), Inches(0.6))
    p2 = tb2.text_frame.paragraphs[0]
    run2 = p2.add_run()
    run2.text = f"Auto-generated summary  ·  {d.get('meeting_date', '')}  ·  REQUIRES HUMAN REVIEW"
    run2.font.size = Pt(12)
    run2.font.color.rgb = RGBColor(0xAA, 0xC4, 0xE0)

    tb3 = slide.shapes.add_textbox(Inches(0.8), Inches(5.2), Inches(8.4), Inches(0.6))
    p3 = tb3.text_frame.paragraphs[0]
    run3 = p3.add_run()
    run3.text = "Participants: " + "  ·  ".join(d.get("participants", []))
    run3.font.size = Pt(11)
    run3.font.color.rgb = RGBColor(0xCC, 0xDD, 0xEE)


def add_summary_slide(prs, d):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide_header(slide, prs, "Executive Summary")

    tb = slide.shapes.add_textbox(Inches(0.6), Inches(1.5), Inches(8.8), Inches(4.5))
    tf = tb.text_frame
    tf.word_wrap = True
    p = tf.paragraphs[0]
    run = p.add_run()
    run.text = d.get("executive_summary", "No summary available.")
    run.font.size = Pt(14)
    run.font.color.rgb = DARK_TEXT

    note = d.get("confidence_note")
    if note:
        p2 = tf.add_paragraph()
        p2.space_before = Pt(20)
        run2 = p2.add_run()
        run2.text = f"AI note: {note}"
        run2.font.size = Pt(11)
        run2.font.italic = True
        run2.font.color.rgb = MID_GREY


def add_objectives_slide(prs, d):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide_header(slide, prs, "High-Level Objectives")

    top = Inches(1.5)
    card_h = Inches(1.4)
    gap = Inches(0.17)

    for i, obj in enumerate(d.get("objectives", [])[:3]):
        card = slide.shapes.add_shape(
            1, Inches(0.6), top + i * (card_h + gap), Inches(8.8), card_h
        )
        card.fill.solid()
        card.fill.fore_color.rgb = LIGHT_GREY
        card.line.color.rgb = RGBColor(0xDD, 0xDD, 0xDD)

        tb = slide.shapes.add_textbox(
            Inches(0.9), top + i * (card_h + gap) + Inches(0.12),
            Inches(8.2), card_h - Inches(0.25)
        )
        tf = tb.text_frame
        tf.word_wrap = True

        p = tf.paragraphs[0]
        r1 = p.add_run()
        r1.text = f"0{i+1}  "
        r1.font.bold = True
        r1.font.size = Pt(13)
        r1.font.color.rgb = ACCENT_BLUE

        r2 = p.add_run()
        r2.text = obj.get("title", "")
        r2.font.bold = True
        r2.font.size = Pt(13)
        r2.font.color.rgb = DARK_TEXT

        p2 = tf.add_paragraph()
        r3 = p2.add_run()
        r3.text = obj.get("detail", "")
        r3.font.size = Pt(11)
        r3.font.color.rgb = DARK_TEXT


def add_actions_slide(prs, d):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide_header(slide, prs, "Actionable Items")

    top = Inches(1.5)
    row_h = Inches(1.25)
    col_x = [Inches(0.5), Inches(2.8), Inches(7.6)]
    col_w = [Inches(2.2), Inches(4.7), Inches(1.9)]

    for j, label in enumerate(["Owner", "Action", "Due"]):
        hb = slide.shapes.add_shape(1, col_x[j], top, col_w[j], Inches(0.4))
        hb.fill.solid()
        hb.fill.fore_color.rgb = ACCENT_BLUE
        hb.line.fill.background()
        tb = slide.shapes.add_textbox(
            col_x[j] + Inches(0.1), top + Inches(0.05), col_w[j], Inches(0.3)
        )
        p = tb.text_frame.paragraphs[0]
        run = p.add_run()
        run.text = label
        run.font.bold = True
        run.font.size = Pt(11)
        run.font.color.rgb = WHITE

    for i, action in enumerate(d.get("action_items", [])[:3]):
        y = top + Inches(0.4) + i * row_h
        bg = LIGHT_GREY if i % 2 == 0 else WHITE
        for j, key in enumerate(["owner", "action", "due"]):
            cb = slide.shapes.add_shape(1, col_x[j], y, col_w[j], row_h)
            cb.fill.solid()
            cb.fill.fore_color.rgb = bg
            cb.line.color.rgb = RGBColor(0xDD, 0xDD, 0xDD)
            tb = slide.shapes.add_textbox(
                col_x[j] + Inches(0.1), y + Inches(0.1),
                col_w[j] - Inches(0.2), row_h - Inches(0.2)
            )
            tf = tb.text_frame
            tf.word_wrap = True
            p = tf.paragraphs[0]
            run = p.add_run()
            run.text = str(action.get(key, "—"))
            run.font.size = Pt(11)
            run.font.color.rgb = DARK_TEXT


def add_next_steps_slide(prs, d):
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    slide_header(slide, prs, "Next Steps & Risks")

    tb = slide.shapes.add_textbox(Inches(0.6), Inches(1.4), Inches(8.8), Inches(2.8))
    tf = tb.text_frame
    tf.word_wrap = True
    for i, step in enumerate(d.get("next_steps", [])):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.space_before = Pt(5)
        run = p.add_run()
        run.text = f"  {i+1}.  {step}"
        run.font.size = Pt(13)
        run.font.color.rgb = DARK_TEXT

    risks = d.get("risks_and_flags", [])
    if risks:
        tb2 = slide.shapes.add_textbox(Inches(0.6), Inches(4.3), Inches(8.8), Inches(2.2))
        tf2 = tb2.text_frame
        tf2.word_wrap = True
        ph = tf2.paragraphs[0]
        rh = ph.add_run()
        rh.text = "Risks & Flags"
        rh.font.bold = True
        rh.font.size = Pt(12)
        rh.font.color.rgb = RGBColor(0xC0, 0x39, 0x2B)
        for risk in risks:
            pr = tf2.add_paragraph()
            rr = pr.add_run()
            rr.text = f"  •  {risk}"
            rr.font.size = Pt(11)
            rr.font.color.rgb = DARK_TEXT


def generate_pptx(data: dict, output_dir: str) -> str:
    prs = Presentation()
    prs.slide_width  = Inches(10)
    prs.slide_height = Inches(7.5)

    add_title_slide(prs, data)
    add_summary_slide(prs, data)
    add_objectives_slide(prs, data)
    add_actions_slide(prs, data)
    add_next_steps_slide(prs, data)

    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    path = os.path.join(output_dir, f"meeting_summary_{ts}.pptx")
    prs.save(path)
    return path


# Generate the deck
deck_path = generate_pptx(meeting_data, OUTPUT_DIR)
print(f"✅ Deck saved: {deck_path}")

# Clickable download link inside the notebook
display(FileLink(deck_path, result_html_prefix="📎 Download deck: "))

✅ Deck saved: ./AiAutomation\meeting_summary_20260410_0756.pptx


C:\Users\rober\Downloads\AiAutomation\meeting_summary_20260410_0756.pptx

---
## Cell 8 — Preview Email Draft

In [15]:
def generate_email_draft(d: dict) -> str:
    lines = [
        f"Subject: Meeting Summary – {d.get('meeting_title', 'Leadership Call')} [{d.get('meeting_date', '')}]",
        "",
        "Hi team,",
        "",
        "Please find below the auto-generated summary of today's meeting. "
        "This has been reviewed and approved for distribution.",
        "",
        "─── Executive Summary ─────────────────────────────────────────────────",
        d.get("executive_summary", ""),
        "",
        "─── Key Objectives ────────────────────────────────────────────────────",
    ]
    for i, obj in enumerate(d.get("objectives", []), 1):
        lines.append(f"  {i}. {obj.get('title', '')} — {obj.get('detail', '')}")

    lines += ["", "─── Action Items ──────────────────────────────────────────────────────"]
    for item in d.get("action_items", []):
        lines.append(
            f"  • [{item.get('owner', '?')}] {item.get('action', '')}  (Due: {item.get('due', 'TBD')})"
        )

    lines += ["", "─── Next Steps ────────────────────────────────────────────────────────"]
    for step in d.get("next_steps", []):
        lines.append(f"  • {step}")

    lines += [
        "",
        "The full deck is attached. Please review and raise any questions in #leadership-ops.",
        "",
        "This email was auto-generated by the BrightLane Meeting Intelligence workflow.",
        "Questions about output quality → Marcus Lee (Ops).",
    ]
    return "\n".join(lines)


email_draft = generate_email_draft(meeting_data)

# Render as formatted HTML box
display(HTML(f"""
<div style='font-family:monospace;font-size:13px;background:#f8f8f8;
            border:1px solid #ddd;border-radius:6px;padding:20px;
            white-space:pre-wrap;max-width:860px;line-height:1.6'>
<div style='font-family:Arial;font-size:11px;color:#888;margin-bottom:12px'>
  ⚠️  DRAFT — requires human review and approval before sending
</div>
{email_draft}
</div>
"""))

---
## Cell 9 — Save Audit Log

Saves the full structured JSON for traceability — so anyone can ask *"why did the AI say this?"* and trace it back.

In [17]:
ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")
audit_path = os.path.join(OUTPUT_DIR, f"audit_log_{ts}.json")

with open(audit_path, "w", encoding="utf-8") as f:
    json.dump(meeting_data, f, indent=2, ensure_ascii=False)

print(f"✅ Audit log saved: {audit_path}")
display(FileLink(audit_path, result_html_prefix="📋 Download audit log: "))

print("\n" + "="*60)
print("Pipeline complete.")
print(f"  Deck  → {deck_path}")
print(f"  Audit → {audit_path}")
print("="*60)
print("\n⚠️  HUMAN REVIEW REQUIRED before distributing any outputs.")

✅ Audit log saved: ./AiAutomation\audit_log_20260410_0757.json


C:\Users\rober\Downloads\AiAutomation\audit_log_20260410_0757.json


Pipeline complete.
  Deck  → ./AiAutomation\meeting_summary_20260410_0756.pptx
  Audit → ./AiAutomation\audit_log_20260410_0757.json

⚠️  HUMAN REVIEW REQUIRED before distributing any outputs.
